# 02. BM25 Mechanics

Coming off [NB1](./01_introduction.ipynb): you've seen the scoring intuition, the `bm25s` API in five calls, and the vocabulary-mismatch teaser at the end. That's the happy path.

**This is the walk.** Now the knobs. Tokenization options, the `k1` and `b` parameters, the failure modes BM25 has built-in answers for (stemming) and the ones it doesn't (synonyms, conceptual paraphrases). Plus how to persist an index. Still no LLM.

## Setup

In [1]:
import bm25s
import pandas as pd
from helpers import build_bm25_index, retrieve
from scripts.build_corpus import build_if_missing

build_if_missing()
df = pd.read_parquet("data/corpus.parquet")

# build_bm25_index is the helper version of the three-line dance from NB1:
# tokens = bm25s.tokenize(docs); model = bm25s.BM25(); model.index(tokens)
retriever, tokens = build_bm25_index(df["text"].tolist())

print(f"Loaded {len(df)} tips, indexed.")

Loaded 12 tips, indexed.


## Tokenization, demystified

NB1 showed that `bm25s.tokenize` strips stopwords and lowercases by default. Let's look at those choices side by side.

In [2]:
# Default: lowercases, removes English stopwords ("the", "is", "and"...)
default = bm25s.tokenize(["the cat is on the mat and it is sleeping"], show_progress=False)
print("Default tokenizer:")
print(default)

Default tokenizer:
Tokenized(
  "ids": [
    0: [0, 1, 2]
  ],
  "vocab": [
    'cat': 0
    'mat': 1
    'sleeping': 2
  ],
)


In [3]:
# No stopword removal: every token kept
keep_all = bm25s.tokenize(
    ["the cat is on the mat and it is sleeping"],
    stopwords=None,
    show_progress=False,
)
print("Stopwords disabled:")
print(keep_all)

Stopwords disabled:
Tokenized(
  "ids": [
    0: [0, 1, 2, 3, 0, 4, 5, 6, 2, 7]
  ],
  "vocab": [
    'and': 5
    'cat': 1
    'is': 2
    'it': 6
    'mat': 4
    'on': 3
    'sleeping': 7
    'the': 0
  ],
)


Stopwords are great for English prose because words like "the" and "is" appear in almost every document, contributing nothing to the ranking. Removing them shrinks the vocabulary and makes the IDF math cleaner. The default English list catches the obvious ones.

When you wouldn't want stopword removal: corpora where short function words *are* the signal (code search, log search, anything with high-information short tokens like `if`/`return`/`error`).

## Stemming

By default `bm25s` does no stemming, so "sort", "sorted", "sorting", and "sorts" are four different tokens. Stemming collapses inflected forms back to a root, which raises recall at the cost of some precision.

In [4]:
import Stemmer

stemmer = Stemmer.Stemmer("english")
stemmed = bm25s.tokenize(
    ["sorting sorted sort sorts"],
    stemmer=stemmer,
    show_progress=False,
)
print("With English Snowball stemmer:")
print(stemmed)

With English Snowball stemmer:
Tokenized(
  "ids": [
    0: [0, 0, 0, 0]
  ],
  "vocab": [
    'sort': 0
  ],
)


All four forms collapsed to one token. Good when your users say "sorted" and your docs say "sort" (or vice versa). Less good when the morphological difference matters: in code search, `running` (the gerund) and `runs` (the function call) probably shouldn't collapse.

The portfolio honest take: defaults are usually fine for English prose, including the no-stemmer default. Reach for stemming when your corpus is small and your users are conversational.

## The `k1` and `b` parameters

BM25 has two tunable parameters:

- **`k1`** (default `1.5`) controls TF saturation. Higher `k1` rewards repeated matches more strongly; lower `k1` reaches saturation faster, treating two matches almost the same as one.
- **`b`** (default `0.75`) controls length normalization. `b=0` ignores length entirely; `b=1` fully normalizes (a doc that's 2x longer needs to match the query 2x as much to score the same).

Most people never touch these. The defaults are good. But let's actually see them move.

In [5]:
# k1 sweep: same query, three different k1 values
print("Query 'sort' with k1 in {0.1, 1.5, 3.0}, b=0.75:")
for k1 in [0.1, 1.5, 3.0]:
    r, _ = build_bm25_index(df["text"].tolist(), k1=k1, b=0.75)
    idx, scores = retrieve(r, "sort", k=3)
    top = [(df.iloc[i]["id"], round(scores[rank], 3)) for rank, i in enumerate(idx)]
    print(f"  k1={k1}:  {top}")

Query 'sort' with k1 in {0.1, 1.5, 3.0}, b=0.75:
  k1=0.1:  [('tip_01', 1.57), ('tip_02', 1.563), ('tip_12', 0.0)]
  k1=1.5:  [('tip_01', 0.94), ('tip_02', 0.906), ('tip_12', 0.0)]
  k1=3.0:  [('tip_01', 0.658), ('tip_02', 0.625), ('tip_12', 0.0)]


On this 12-doc corpus the ranking stays the same (tip_01, tip_02, tip_12) but the scores compress as `k1` rises. With `k1=0.1`, TF saturates almost immediately, so the top scores are close. With `k1=3.0`, repeated matches keep accumulating reward.

On real corpora with lots of TF-heavy queries, `k1` does shift rankings. For most workloads the default is fine.

In [6]:
# b sweep: length normalization
print("Query 'memory class instances' with b in {0.0, 0.75, 1.0}, k1=1.5:")
for b in [0.0, 0.75, 1.0]:
    r, _ = build_bm25_index(df["text"].tolist(), k1=1.5, b=b)
    idx, scores = retrieve(r, "memory class instances", k=3)
    top = [(df.iloc[i]["id"], round(scores[rank], 3)) for rank, i in enumerate(idx)]
    print(f"  b={b}:   {top}")

Query 'memory class instances' with b in {0.0, 0.75, 1.0}, k1=1.5:
  b=0.0:   [('tip_06', 2.183), ('tip_08', 1.602), ('tip_12', 0.0)]
  b=0.75:   [('tip_06', 2.224), ('tip_08', 1.656), ('tip_12', 0.0)]
  b=1.0:   [('tip_06', 2.238), ('tip_08', 1.675), ('tip_12', 0.0)]


`b` moves scores even less on this corpus because all 12 docs are about the same length (46-67 words). On a corpus with one-line tweets and 10-page articles, `b` matters a lot more.

**When to tune `k1` or `b`: rarely.** If your defaults aren't producing the rankings you want, the problem is almost always upstream (your tokenization, your stopword list, the vocabulary of your corpus vs your queries), not the BM25 hyperparameters.

## BM25 has no concept of word order

This is worth seeing once so you remember it forever. The same tokens in any order produce the same ranking.

In [7]:
for q in [
    "sort list dict",
    "dict list sort",
    "list sort dict",
]:
    idx, scores = retrieve(retriever, q, k=2)
    top = [(df.iloc[i]["id"], round(scores[rank], 3)) for rank, i in enumerate(idx)]
    print(f"  {q!r:>20}: {top}")

      'sort list dict': [('tip_01', 1.364), ('tip_02', 1.308)]
      'dict list sort': [('tip_01', 1.364), ('tip_02', 1.308)]
      'list sort dict': [('tip_01', 1.364), ('tip_02', 1.308)]


Identical. BM25 treats the query as a bag of tokens, so word order is invisible to it. If word order matters in your domain (legal contracts, code, multi-word entities), you need either a phrase-aware retriever (BM25F with field weighting, or a dense embedding model) or a post-filtering step that enforces order on candidate documents BM25 surfaced.

## Where BM25 wins: rare identifiers

This is the sentence to take with you when deciding whether to use BM25 in production: it nails rare distinctive tokens like a knife through butter.

In [8]:
idx, scores = retrieve(retriever, "__slots__", k=3)
for rank, i in enumerate(idx, start=1):
    print(f"  #{rank}  {df.iloc[i]['id']}  score={scores[rank-1]:.3f}  {df.iloc[i]['title']}")

  #1  tip_06  score=0.880  Use __slots__ to shrink instance memory
  #2  tip_12  score=0.000  Speed up loops by avoiding global name lookups
  #3  tip_10  score=0.000  pathlib.Path replaces os.path for new code


tip_06 wins clean. The double-underscore identifier `__slots__` appears in exactly one document, so the IDF reward is huge. Vector embeddings would *also* find this document, but they would also surface several semantically-adjacent ones with similar scores. BM25 just gives you the right one with a big gap behind it.

This is the BM25 strong suit: rare distinctive tokens (identifiers, names, version numbers, error codes, IDs). NB3's head-to-head leans on it.

## Where BM25 loses: synonyms and paraphrase

The mirror image of the win. Same kind of question, but phrased without the document's vocabulary.

In [9]:
idx, scores = retrieve(retriever, "make my code faster", k=3)
for rank, i in enumerate(idx, start=1):
    print(f"  #{rank}  {df.iloc[i]['id']}  score={scores[rank-1]:.3f}  {df.iloc[i]['title']}")

  #1  tip_03  score=0.687  bisect keeps a sorted list sorted on insert
  #2  tip_01  score=0.658  Sort a list of dicts by a key with operator.itemgetter
  #3  tip_04  score=0.437  collections.OrderedDict preserves insertion order


tip_12 ("Speed up loops by avoiding global name lookups") isn't even in the top 3. The user clearly wants that doc. BM25 can't deliver it because the query shares zero meaningful tokens with the document. "Make" and "code" don't appear in tip_12; "faster" doesn't either (the doc says "speed up", "fast", "savings").

This is the **synonym trap**. It's the central failure mode of BM25 on natural language. The fixes:

1. **Stemming** helps if the mismatch is morphological (sort/sorted/sorting), but not when it's lexical (faster/speed up).
2. **Query expansion** (replace "faster" with "fast OR speed up OR quick") can paper over it, but you need a synonym dictionary.
3. **Vector retrieval** solves it natively, but at the cost of needing embeddings and losing BM25's exact-match precision.
4. **Hybrid retrieval** combines BM25 and vectors. Usually beats either alone.

NB3 does the head-to-head and walks through option 4.

## Filtering by metadata

`bm25s` doesn't have metadata filtering built in. The practical pattern is to retrieve a larger candidate set and then filter the result in pandas.

In [10]:
# Retrieve top 10, then keep only tips whose title mentions "dict"
idx, scores = retrieve(retriever, "preserves order", k=10)
candidates = pd.DataFrame({
    "id": [df.iloc[i]["id"] for i in idx],
    "title": [df.iloc[i]["title"] for i in idx],
    "score": scores,
})
candidates[candidates["title"].str.contains("dict", case=False)]

,id,title,score
0,tip_05,dict itself preserves insertion order since Py...,1.693695
1,tip_04,collections.OrderedDict preserves insertion order,0.765652


Same idea works for any column. If you needed real performance, you'd push the filter down into the indexing layer (`bm25s` doesn't support that, but Pinecone and tantivy do). For learning workloads, post-filtering is fine.

## Persistence

Indexing 12 docs is instant, but indexing a million docs isn't. Save the index and reload it next time.

In [11]:
retriever.save("data/bm25_index")
print("Saved.")

Saved.


In [12]:
reloaded = bm25s.BM25.load("data/bm25_index", load_corpus=False)
idx, scores = retrieve(reloaded, "__slots__", k=3)
for rank, i in enumerate(idx, start=1):
    print(f"  #{rank}  {df.iloc[i]['id']}  score={scores[rank-1]:.3f}  {df.iloc[i]['title']}")

  #1  tip_06  score=0.880  Use __slots__ to shrink instance memory
  #2  tip_12  score=0.000  Speed up loops by avoiding global name lookups
  #3  tip_10  score=0.000  pathlib.Path replaces os.path for new code


Same scores as before. The index round-trips cleanly. In production you'd save your index alongside your model artifacts and reload it on startup.

## Recap

| Topic | Default | When to change |
|---|---|---|
| Stopwords | English list | Code/log search, short-token domains |
| Stemming | Off | Small corpus, conversational queries |
| `k1` (TF saturation) | 1.5 | TF-heavy domains with repeated keywords |
| `b` (length normalization) | 0.75 | Mixed-length corpora |
| Phrase queries | Not supported (bag of words) | Use a phrase-aware retriever or post-filter |
| Metadata filters | Not built in | Post-filter in pandas; or use Pinecone/tantivy |

Where BM25 wins: **rare distinctive tokens** (identifiers, names, version numbers, error codes). Where it loses: **synonyms and paraphrase** (same intent, different vocabulary).

**Next:** [NB3 - 03. Vectorless RAG](./03_vectorless_rag.ipynb) builds the full RAG loop with `gpt-4o-mini`, then does the head-to-head against module 01's Pinecone index on the same questions, then resolves the synonym trap with hybrid retrieval via Reciprocal Rank Fusion.